# CLIP matching

This notebook contains the code for CLIP matching used to construct the Hard Benchmark. 

In [ ]:
%pip install clip-retrieval open-clip-torch requests pillow

In [ ]:
import os
import shutil
import torch
import open_clip
from PIL import Image
from tqdm.auto import tqdm

# Config
# ==========================================
REAL_POOL_DIR = "Benchmark_Reals"
FAKE_ROOT_DIR = "Benchmark_Fakes/Benchmark_RAW"
OUTPUT_ROOT   = "Matched_Reals"

EMBED_CACHE   = "FLOWTRACER/real_pool_embeddings.pt"
MASK_CACHE    = "FLOWTRACER/available_mask.pt"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

DATASETS = ['SD3', 'Midjourney_v6', 'DALL_E_3', 'Nano_Banana', 'FLUX', 'WildRF']

# Setup CLIP Model
# ======================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP model on {device}...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14', pretrained='laion2b_s32b_b82k'
)
clip_model.eval()
clip_model.to(device)

# Load Real Embeddings
# ==============================
if os.path.exists(EMBED_CACHE):
    # Resume function
    print(f"Loading cache...")
    cache          = torch.load(EMBED_CACHE, weights_only=True)
    real_tensor    = cache['embeddings']
    real_filepaths = cache['filepaths']
    print(f"   Loaded {len(real_filepaths):,} embeddings from cache.")
else:
    # Get embeddings and save
    real_files = []
    for root, _, files in os.walk(REAL_POOL_DIR):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                real_files.append(os.path.join(root, f))

    print(f"Computing CLIP embeddings...")

    real_embeddings = []
    real_filepaths  = []

    with torch.no_grad():
        for path in tqdm(real_files, desc="Embedding Reals"):
            try:
                raw        = Image.open(path).convert('RGB')
                img_tensor = clip_preprocess(raw).unsqueeze(0).to(device)
                emb        = clip_model.encode_image(img_tensor)
                emb        = emb / emb.norm(dim=-1, keepdim=True)
                real_embeddings.append(emb.cpu())
                real_filepaths.append(path)
            except Exception:
                continue

    real_tensor = torch.cat(real_embeddings, dim=0)  # [N_reals, 768]

    print(f"Saving embedding cache to Drive...")
    torch.save({'embeddings': real_tensor, 'filepaths': real_filepaths}, EMBED_CACHE)
    print(f"Done.")

# Load Available Mask
# ===========================
if os.path.exists(MASK_CACHE):
    # Resume function
    available_mask = torch.load(MASK_CACHE, weights_only=True)
    used_count     = (~available_mask).sum().item()
    print(f"\n Resuming from mask cache. {used_count:,} already matched, "
          f"{available_mask.sum().item():,} still available.")
else:
    # FIRST RUN: Every image available
    available_mask = torch.ones(len(real_filepaths), dtype=torch.bool)
    print(f"\nFresh run. {len(real_filepaths):,} real images available.")

# Summary Check
# ===================
total_fakes = sum(
    len([f for f in os.listdir(os.path.join(FAKE_ROOT_DIR, gen))
         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
    for gen in DATASETS
    if os.path.exists(os.path.join(FAKE_ROOT_DIR, gen))
)
print(f"Total fakes : {total_fakes:,}")
print(f"Real pool size : {len(real_filepaths):,}")

# Match Fakes to Reals (No Duplicates)
# ======================================================
with torch.no_grad():
    for gen in DATASETS:
        fake_dir = os.path.join(FAKE_ROOT_DIR, gen)
        if not os.path.exists(fake_dir):
            fake_dir = os.path.join(FAKE_ROOT_DIR, gen.replace('_', '-'))
        if not os.path.exists(fake_dir):
            print(f"\n{gen} not found.")
            continue

        output_dir = os.path.join(OUTPUT_ROOT, gen)
        os.makedirs(output_dir, exist_ok=True)

        fake_files = sorted([
            f for f in os.listdir(fake_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
        ])
        print(f"\nMatching {len(fake_files):,} fakes for {gen}...")

        for fake_filename in tqdm(fake_files, desc=f"Matching {gen}"):
            fake_path = os.path.join(fake_dir, fake_filename)
            fake_stem = os.path.splitext(fake_filename)[0]
            save_path = os.path.join(output_dir, f"real_{fake_stem}.jpg")

            # Skip already-matched images
            if os.path.exists(save_path):
                continue

            try:
                # Embed fake image
                raw        = Image.open(fake_path).convert('RGB')
                img_tensor = clip_preprocess(raw).unsqueeze(0).to(device)
                fake_emb   = clip_model.encode_image(img_tensor)
                fake_emb   = (fake_emb / fake_emb.norm(dim=-1, keepdim=True)).cpu()

                # Score against unused images
                similarities = torch.matmul(fake_emb, real_tensor.T).squeeze()
                similarities[~available_mask] = -torch.inf

                best_idx = torch.argmax(similarities).item()

                # Mark as used
                available_mask[best_idx] = False

                # Copy matched image to output
                shutil.copy2(real_filepaths[best_idx], save_path)

            except Exception as e:
                print(f"  Error on {fake_filename}: {e}")

        # Save mask after every generator
        torch.save(available_mask, MASK_CACHE)
        remaining = available_mask.sum().item()
        print(f"  {gen} done. {remaining:,}")

print(f"\nDone.")
print(f"Output: {OUTPUT_ROOT}")